# Module 02: Transformers from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/02-transformers/notebook.ipynb)

**GPU recommended:** No (trains on CPU in ~2 minutes). Use GPU for larger models.

## Setup and Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

plt.style.use("seaborn-v0_8-whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 1. Scaled Dot-Product Attention from Scratch

The core operation of the transformer is scaled dot-product attention:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

where $Q$, $K$, $V$ are the query, key, and value matrices, and $d_k$ is the dimension of the keys.

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """Compute scaled dot-product attention.

    Args:
        query: (batch, ..., seq_len_q, d_k)
        key:   (batch, ..., seq_len_k, d_k)
        value: (batch, ..., seq_len_k, d_v)
        mask:  broadcastable to (batch, ..., seq_len_q, seq_len_k)

    Returns:
        output:  (batch, ..., seq_len_q, d_v)
        weights: (batch, ..., seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, value)
    return output, weights

### Test with a small example

In [ ]:
torch.manual_seed(42)

seq_len, d_k, d_v = 6, 8, 8
query = torch.randn(1, seq_len, d_k)
key = torch.randn(1, seq_len, d_k)
value = torch.randn(1, seq_len, d_v)

output, weights = scaled_dot_product_attention(query, key, value)
print(f"Output shape:  {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"Weights sum per query (should be 1.0): {weights.sum(dim=-1)}")

### Visualize attention weights

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights[0].detach().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title("Scaled Dot-Product Attention Weights")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Causal (autoregressive) mask

In [ ]:
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)  # (1, seq_len, seq_len)
print("Causal mask:")
print(causal_mask[0])

output_causal, weights_causal = scaled_dot_product_attention(query, key, value, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, w, title in zip(
    axes,
    [weights[0], weights_causal[0]],
    ["Without mask", "With causal mask"],
):
    im = ax.imshow(w.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---
## 2. Multi-Head Attention

Multi-head attention runs several attention operations in parallel, each with
its own learned projection, then concatenates and projects the results:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.attn_weights = None  # stored for visualization

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Linear projections and reshape to (batch, num_heads, seq_len, d_k)
        q = self.w_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Expand mask for heads dimension: (batch, 1, seq_q, seq_k)
        if mask is not None and mask.dim() == 3:
            mask = mask.unsqueeze(1)

        # Scaled dot-product attention per head
        output, weights = scaled_dot_product_attention(q, k, v, mask=mask)
        self.attn_weights = weights.detach()  # (batch, num_heads, seq_q, seq_k)

        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(output)

### Verify shapes

In [ ]:
torch.manual_seed(42)

d_model = 64
num_heads = 4
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)

out = mha(x, x, x, mask=causal_mask)

print(f"Input shape:            {x.shape}")
print(f"Output shape:           {out.shape}")
print(f"Attention weights shape: {mha.attn_weights.shape}")
assert out.shape == (batch_size, seq_len, d_model)
assert mha.attn_weights.shape == (batch_size, num_heads, seq_len, seq_len)
print("All shape checks passed.")

---
## 3. Sinusoidal Positional Encodings

Transformers have no built-in notion of order, so we inject position
information with sinusoidal encodings:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Generate sinusoidal positional encodings.

    Returns:
        Tensor of shape (max_len, d_model)
    """
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

In [ ]:
pe = sinusoidal_positional_encoding(max_len=128, d_model=64)
print(f"Positional encoding shape: {pe.shape}")

### Heatmap visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe.numpy().T, cmap="RdBu", aspect="auto", origin="lower")
ax.set_xlabel("Position")
ax.set_ylabel("Encoding dimension")
ax.set_title("Sinusoidal Positional Encodings")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Positional similarity

Nearby positions should have higher dot-product similarity than distant ones.

In [ ]:
pe_normed = pe / pe.norm(dim=-1, keepdim=True)
similarity = torch.matmul(pe_normed, pe_normed.T)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(similarity[:50, :50].numpy(), cmap="viridis", aspect="auto")
ax.set_xlabel("Position")
ax.set_ylabel("Position")
ax.set_title("Positional Encoding Dot-Product Similarity (first 50)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---
## 4. Transformer Block

A single transformer decoder block follows this pattern (Pre-LN variant):

```
x -> LayerNorm -> Multi-Head Self-Attention -> + residual
                                                |
                                                v
                            LayerNorm -> FFN -> + residual
```

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN: LayerNorm before attention
        normed = self.ln1(x)
        x = x + self.dropout1(self.attn(normed, normed, normed, mask=mask))

        # Pre-LN: LayerNorm before FFN
        normed = self.ln2(x)
        x = x + self.dropout2(self.ff(normed))
        return x

### Verify the block

In [ ]:
torch.manual_seed(42)

block = TransformerBlock(d_model=64, num_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)
mask = torch.tril(torch.ones(10, 10)).unsqueeze(0)

out = block(x, mask=mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape
print("TransformerBlock shape check passed.")

---
## 5. Full Decoder-Only Transformer (GPT-style)

The full model stacks token embeddings, positional embeddings, multiple
transformer blocks, and a language model head that projects back to the
vocabulary.

In [ ]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_seq_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying between token embedding and LM head
        self.lm_head.weight = self.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        """Forward pass.

        Args:
            idx: (batch, seq_len) token indices

        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = idx.shape
        assert seq_len <= self.max_seq_len, f"Sequence length {seq_len} exceeds max {self.max_seq_len}"

        positions = torch.arange(seq_len, device=idx.device).unsqueeze(0)
        x = self.dropout(self.token_emb(idx) + self.pos_emb(positions))

        # Causal mask
        mask = torch.tril(torch.ones(seq_len, seq_len, device=idx.device)).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask=mask)

        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits

### Verify the full model

In [ ]:
torch.manual_seed(42)

model = DecoderOnlyTransformer(
    vocab_size=16,
    d_model=64,
    num_heads=4,
    d_ff=256,
    num_layers=2,
    max_seq_len=32,
)

dummy_input = torch.randint(0, 16, (2, 12))
logits = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Logits shape: {logits.shape}")
assert logits.shape == (2, 12, 16)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print("Full model shape check passed.")

---
## 6. Toy Task: Learn to Reverse Short Digit Sequences

We train the model to reverse short sequences of digits. For example:

```
Input:  [BOS] 3 1 4 [SEP] ? ? ? [EOS]
Target:  -    - - -   -   4 1 3 [EOS]
```

The model learns to predict the reversed sequence after the `[SEP]` token.

### Vocabulary

- Tokens 0-9: digits
- Token 10: `[PAD]`
- Token 11: `[BOS]`
- Token 12: `[SEP]`
- Token 13: `[EOS]`

In [ ]:
PAD_TOKEN = 10
BOS_TOKEN = 11
SEP_TOKEN = 12
EOS_TOKEN = 13
VOCAB_SIZE = 14

TOKEN_NAMES = {i: str(i) for i in range(10)}
TOKEN_NAMES.update({PAD_TOKEN: "[PAD]", BOS_TOKEN: "[BOS]", SEP_TOKEN: "[SEP]", EOS_TOKEN: "[EOS]"})


def tokens_to_str(tokens):
    """Convert a list of token ids to a readable string."""
    return " ".join(TOKEN_NAMES.get(t, f"?{t}") for t in tokens)

In [ ]:
def generate_reversal_dataset(num_samples, min_len=2, max_len=5):
    """Generate sequences for the reversal task.

    Each sample: [BOS] d1 d2 ... dn [SEP] dn ... d2 d1 [EOS]

    Returns:
        inputs: tensor of shape (num_samples, max_total_len)
        targets: tensor of shape (num_samples, max_total_len)
            with -100 for positions we don't compute loss on
    """
    # max total length: BOS + max_len digits + SEP + max_len digits + EOS
    max_total_len = 1 + max_len + 1 + max_len + 1

    all_inputs = []
    all_targets = []

    for _ in range(num_samples):
        length = torch.randint(min_len, max_len + 1, (1,)).item()
        digits = torch.randint(0, 10, (length,)).tolist()
        reversed_digits = list(reversed(digits))

        seq = [BOS_TOKEN] + digits + [SEP_TOKEN] + reversed_digits + [EOS_TOKEN]

        # Target: we only predict from after the SEP token onward
        # For a language model, target is the next token at each position
        target = [-100] * (1 + length) + reversed_digits + [EOS_TOKEN] + [-100]
        # target[i] is the desired prediction for input position i
        # Actually, for next-token prediction: target[i] = seq[i+1]
        # But we only care about positions after SEP
        target_shifted = [-100] * (1 + length + 1) + reversed_digits + [EOS_TOKEN]
        # This means: at position of SEP, predict first reversed digit, etc.

        # Pad to max_total_len
        pad_len = max_total_len - len(seq)
        seq = seq + [PAD_TOKEN] * pad_len
        target_shifted = target_shifted + [-100] * pad_len

        all_inputs.append(seq)
        all_targets.append(target_shifted)

    return torch.tensor(all_inputs), torch.tensor(all_targets)

In [ ]:
torch.manual_seed(42)

train_inputs, train_targets = generate_reversal_dataset(3000)
val_inputs, val_targets = generate_reversal_dataset(500)

print(f"Training set:   {train_inputs.shape}")
print(f"Validation set: {val_inputs.shape}")
print()

# Show a few examples
for i in range(5):
    inp = train_inputs[i].tolist()
    tgt = train_targets[i].tolist()
    print(f"Input:  {tokens_to_str(inp)}")
    tgt_display = [TOKEN_NAMES.get(t, "_") if t != -100 else "_" for t in tgt]
    print(f"Target: {' '.join(tgt_display)}")
    print()

### Training loop

In [ ]:
torch.manual_seed(42)

model = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=64,
    num_heads=4,
    d_ff=256,
    num_layers=4,
    max_seq_len=train_inputs.shape[1],
    dropout=0.1,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Sequence length:  {train_inputs.shape[1]}")
print(f"Training samples: {len(train_inputs)}")

In [ ]:
import time

batch_size = 64
num_epochs = 30
train_losses = []
val_losses = []
val_accuracies = []

train_inputs_dev = train_inputs.to(device)
train_targets_dev = train_targets.to(device)
val_inputs_dev = val_inputs.to(device)
val_targets_dev = val_targets.to(device)

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    # Shuffle training data
    perm = torch.randperm(len(train_inputs_dev))
    train_inputs_shuffled = train_inputs_dev[perm]
    train_targets_shuffled = train_targets_dev[perm]

    for i in range(0, len(train_inputs_dev), batch_size):
        batch_in = train_inputs_shuffled[i:i + batch_size]
        batch_tgt = train_targets_shuffled[i:i + batch_size]

        logits = model(batch_in)

        # Shift: predict next token. logits[:, :-1] predicts targets[:, 1:]
        # But our targets are already aligned: target[i] is what position i should predict
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            batch_tgt.view(-1),
            ignore_index=-100,
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(val_inputs_dev)
        val_loss = F.cross_entropy(
            val_logits.view(-1, VOCAB_SIZE),
            val_targets_dev.view(-1),
            ignore_index=-100,
        ).item()
        val_losses.append(val_loss)

        # Compute accuracy on non-masked positions
        preds = val_logits.argmax(dim=-1)  # (batch, seq_len)
        valid_mask = val_targets_dev != -100
        correct = (preds == val_targets_dev) & valid_mask
        accuracy = correct.sum().item() / valid_mask.sum().item()
        val_accuracies.append(accuracy)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(
            f"Epoch {epoch+1:3d}/{num_epochs} | "
            f"Train loss: {avg_train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val acc: {accuracy:.4f} | "
            f"Time: {elapsed:.1f}s"
        )

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s")
print(f"Final validation accuracy: {val_accuracies[-1]:.4f}")

### Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, num_epochs + 1)

ax1.plot(epochs_range, train_losses, label="Train", linewidth=1.5)
ax1.plot(epochs_range, val_losses, label="Validation", linewidth=1.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training and Validation Loss")
ax1.legend()

ax2.plot(epochs_range, val_accuracies, color="green", linewidth=1.5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Token Accuracy")
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

### Evaluate: full-sequence accuracy

A sample counts as correct only if every token in the reversed portion is predicted correctly.

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(val_inputs_dev)
    preds = val_logits.argmax(dim=-1)

    valid_mask = val_targets_dev != -100
    # For each sample, check if ALL valid positions are correct
    correct_per_pos = (preds == val_targets_dev) | ~valid_mask
    seq_correct = correct_per_pos.all(dim=-1)
    seq_accuracy = seq_correct.float().mean().item()

print(f"Full-sequence accuracy on validation set: {seq_accuracy:.4f} ({seq_correct.sum().item()}/{len(val_inputs)})")

---
## 7. Autoregressive Generation

At inference time, we generate one token at a time, feeding each prediction
back as input for the next step.

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens, temperature=1.0, greedy=True):
    """Autoregressive generation from a prompt.

    Args:
        model: the transformer model
        prompt: 1D tensor of token ids
        max_new_tokens: how many tokens to generate
        temperature: sampling temperature (ignored if greedy=True)
        greedy: if True, always pick the argmax token

    Returns:
        List of generated token ids (not including the prompt)
    """
    model.eval()
    generated = prompt.unsqueeze(0).to(device)  # (1, prompt_len)
    new_tokens = []

    for _ in range(max_new_tokens):
        logits = model(generated)  # (1, seq_len, vocab_size)
        next_logits = logits[0, -1]  # (vocab_size,)

        if greedy:
            next_token = next_logits.argmax().item()
        else:
            scaled_logits = next_logits / max(temperature, 1e-8)
            probs = F.softmax(scaled_logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

        new_tokens.append(next_token)
        if next_token == EOS_TOKEN:
            break

        next_tensor = torch.tensor([[next_token]], device=device)
        generated = torch.cat([generated, next_tensor], dim=1)

    return new_tokens

### Greedy decoding examples

In [ ]:
test_cases = [
    [3, 1, 4],
    [7, 2],
    [9, 0, 5, 8],
    [1, 2, 3, 4, 5],
    [6, 6, 6],
]

print("Greedy decoding:")
print("-" * 55)
for digits in test_cases:
    prompt = torch.tensor([BOS_TOKEN] + digits + [SEP_TOKEN])
    generated = generate(model, prompt, max_new_tokens=len(digits) + 1, greedy=True)
    expected = list(reversed(digits)) + [EOS_TOKEN]
    match = generated == expected
    status = "PASS" if match else "FAIL"
    print(
        f"  Input: {digits}  "
        f"Expected: {list(reversed(digits))}  "
        f"Got: {[t for t in generated if t != EOS_TOKEN]}  "
        f"[{status}]"
    )

### Temperature sampling

Higher temperature produces more diverse (but potentially less accurate) outputs.

In [ ]:
torch.manual_seed(42)

digits = [5, 3, 8]
prompt = torch.tensor([BOS_TOKEN] + digits + [SEP_TOKEN])

print(f"Input: {digits}")
print(f"Expected reversal: {list(reversed(digits))}")
print()

for temp in [0.1, 0.5, 1.0, 2.0]:
    print(f"Temperature = {temp}:")
    for trial in range(5):
        gen = generate(model, prompt, max_new_tokens=len(digits) + 1,
                       temperature=temp, greedy=False)
        output = [t for t in gen if t != EOS_TOKEN]
        print(f"    {output}")
    print()

---
## 8. Attention Pattern Visualization

We can inspect what each attention head learns by looking at the attention
weight matrices for a specific input.

In [ ]:
def get_attention_maps(model, input_ids):
    """Run a forward pass and collect attention weights from all layers.

    Returns:
        List of tensors, one per layer, each of shape (num_heads, seq_len, seq_len)
    """
    model.eval()
    with torch.no_grad():
        _ = model(input_ids.unsqueeze(0).to(device))

    attention_maps = []
    for block in model.blocks:
        # attn_weights: (1, num_heads, seq_len, seq_len)
        attention_maps.append(block.attn.attn_weights[0].cpu())
    return attention_maps

In [ ]:
sample_digits = [3, 1, 4]
sample_input = torch.tensor([BOS_TOKEN] + sample_digits + [SEP_TOKEN] + list(reversed(sample_digits)) + [EOS_TOKEN])
token_labels = [TOKEN_NAMES[t.item()] for t in sample_input]

print(f"Sequence: {' '.join(token_labels)}")
print(f"Sequence length: {len(sample_input)}")

attn_maps = get_attention_maps(model, sample_input)
print(f"Number of layers: {len(attn_maps)}")
print(f"Attention map shape per layer: {attn_maps[0].shape}")

### All heads across all layers

In [ ]:
num_layers = len(attn_maps)
num_heads = attn_maps[0].shape[0]

fig, axes = plt.subplots(num_layers, num_heads, figsize=(3.5 * num_heads, 3 * num_layers))

for layer_idx in range(num_layers):
    for head_idx in range(num_heads):
        ax = axes[layer_idx, head_idx] if num_layers > 1 else axes[head_idx]
        weights = attn_maps[layer_idx][head_idx].numpy()
        im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1, aspect="auto")

        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(token_labels)))
        ax.set_yticklabels(token_labels, fontsize=7)

        ax.set_title(f"Layer {layer_idx + 1}, Head {head_idx + 1}", fontsize=9)

fig.suptitle("Attention Patterns Across All Layers and Heads", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Attention from the output positions

The most interesting patterns are in the positions after `[SEP]`, where the model
must attend to the correct input digit to produce the reversal.

In [ ]:
# Find the SEP position and show attention from output positions
sep_pos = (sample_input == SEP_TOKEN).nonzero(as_tuple=True)[0].item()
output_positions = list(range(sep_pos, len(sample_input)))

# Average attention across heads in the last layer
last_layer_attn = attn_maps[-1]  # (num_heads, seq_len, seq_len)

fig, axes = plt.subplots(1, num_heads + 1, figsize=(3.5 * (num_heads + 1), 3.5))

for head_idx in range(num_heads):
    ax = axes[head_idx]
    weights = last_layer_attn[head_idx][output_positions].numpy()
    im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=weights.max(), aspect="auto")
    ax.set_xticks(range(len(token_labels)))
    ax.set_xticklabels(token_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(output_positions)))
    ax.set_yticklabels([token_labels[p] for p in output_positions], fontsize=8)
    ax.set_title(f"Head {head_idx + 1}", fontsize=10)
    ax.set_xlabel("Attends to")
    if head_idx == 0:
        ax.set_ylabel("Output position")

# Average across heads
ax = axes[num_heads]
avg_weights = last_layer_attn.mean(dim=0)[output_positions].numpy()
im = ax.imshow(avg_weights, cmap="Blues", vmin=0, vmax=avg_weights.max(), aspect="auto")
ax.set_xticks(range(len(token_labels)))
ax.set_xticklabels(token_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(output_positions)))
ax.set_yticklabels([token_labels[p] for p in output_positions], fontsize=8)
ax.set_title("Average", fontsize=10)
ax.set_xlabel("Attends to")

fig.suptitle("Last Layer: Attention from Output Positions", fontsize=13)
plt.tight_layout()
plt.show()

### Head specialization analysis

Compute the entropy of each head's attention distribution to understand specialization.
Low entropy means the head attends sharply to specific positions; high entropy means diffuse attention.

In [ ]:
def attention_entropy(weights):
    """Compute entropy of attention distributions.

    Args:
        weights: (num_heads, seq_len, seq_len)

    Returns:
        (num_heads,) average entropy per head
    """
    # Clamp to avoid log(0)
    w = weights.clamp(min=1e-12)
    entropy = -(w * w.log()).sum(dim=-1)  # (num_heads, seq_len)
    return entropy.mean(dim=-1)  # (num_heads,)


fig, ax = plt.subplots(figsize=(8, 4))

max_entropy = math.log(len(sample_input))
x_positions = []
x_labels = []
entropies = []

for layer_idx, attn_map in enumerate(attn_maps):
    h = attention_entropy(attn_map)
    for head_idx in range(num_heads):
        x_pos = layer_idx * (num_heads + 1) + head_idx
        x_positions.append(x_pos)
        x_labels.append(f"L{layer_idx+1}H{head_idx+1}")
        entropies.append(h[head_idx].item())

bars = ax.bar(x_positions, entropies, color="steelblue", alpha=0.8)
ax.axhline(y=max_entropy, color="red", linestyle="--", linewidth=1, label=f"Max entropy ({max_entropy:.2f})")
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Average Entropy (nats)")
ax.set_title("Attention Head Entropy (lower = more specialized)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

In this notebook we built a transformer from scratch:

1. **Scaled dot-product attention** -- the core primitive that computes weighted sums of values based on query-key similarity.
2. **Multi-head attention** -- parallel attention operations with independent projections, giving the model capacity to attend to different relationship types simultaneously.
3. **Sinusoidal positional encodings** -- injecting position information so the model can distinguish token order.
4. **Transformer block** -- Pre-LayerNorm architecture with residual connections around attention and feed-forward sub-layers.
5. **Decoder-only transformer** -- stacking everything into a GPT-style autoregressive language model with weight-tied embeddings.
6. **Training on sequence reversal** -- the model learns to reverse short digit sequences, demonstrating that even a small transformer can learn non-trivial input-output mappings.
7. **Autoregressive generation** -- greedy and temperature-based sampling.
8. **Attention visualization** -- inspecting what heads learn and how they specialize.